# DPO training
Train an adapter from the original Qwen2.5-3B-Instruct model using the frozen, verified preference pairs. This notebook does not generate new answers or modify the SFT/GRPO pipelines.

Start with a pilot. Full training uses all 3322 pairs and the separate 500-question MedQA validation set. Training is enabled for the configured full run; execution starts only when the training cell is run.

In [1]:
from pathlib import Path
import sys, os, json, subprocess
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO = Path('/content/drive/MyDrive/medqa-calibration')
else:
    here = Path.cwd().resolve()
    REPO = here if (here / 'train/dpo/train.py').is_file() else here.parent
assert (REPO / 'train/dpo/train.py').is_file(), 'Set REPO to the repository root.'
print('Repository:', REPO)

Mounted at /content/drive
Repository: /content/drive/MyDrive/medqa-calibration


## Dependencies
Activate `DL_Final` before opening this notebook locally. Retain Colab's installed CUDA-enabled PyTorch. Set `INSTALL_DEPENDENCIES=True` only when the DPO dependencies need installation; restart the kernel afterward if the notebook asks for it.

In [2]:
INSTALL_DEPENDENCIES = True
if INSTALL_DEPENDENCIES:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(REPO / 'train/dpo/requirements.txt')])
import importlib.metadata
for name in ['torch', 'transformers', 'trl', 'peft', 'datasets', 'accelerate', 'bitsandbytes']:
    try:
        print(name, importlib.metadata.version(name))
    except importlib.metadata.PackageNotFoundError:
        print(name, 'not installed')

torch 2.14.1+cu130
transformers 4.57.6
trl 1.16.0
peft 0.18.0
datasets 4.8.5
accelerate 1.15.0
bitsandbytes 0.49.2


In [3]:
def run_command(command):
    process = subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace',
                               env=dict(os.environ, PYTHONIOENCODING='utf-8', PYTHONUNBUFFERED='1'))
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
        if process.wait() != 0:
            raise RuntimeError('Command failed; inspect the output above.')
    except BaseException:
        if process.poll() is None:
            process.terminate()
            process.wait()
        raise
    finally:
        process.stdout.close()


## One-time original-model baseline
Run this optional stage before the first training experiment. It evaluates all 500 fixed validation questions and the full official MedQA test split, without adapters or answer generation. Subsequent experiments can leave `RUN_BASELINE=False` and reuse the saved results. Use the same base revision, quantization, and precision for comparable results; if these change, use a separate baseline output directory. Existing outputs are protected from overwrite. A failed partial run requires a new output directory. Test scores must never guide hyperparameter or checkpoint selection.


In [4]:
RUN_BASELINE = False  # Enable for the first run only; later runs display saved results.
BASELINE_SOURCE = 'hf' if IN_COLAB else 'local'
BASELINE_MODEL = 'Qwen/Qwen2.5-3B-Instruct' if BASELINE_SOURCE == 'hf' else str(REPO.parent / 'Qwen2.5-3B-Instruct')
BASELINE_QUANTIZATION = '4bit'
BASELINE_COMPUTE_DTYPE = 'auto'
BASELINE_BATCH_SIZE = 4
BASELINE_OUTPUT = REPO / 'outputs/base_medqa_4bit'


In [5]:
if RUN_BASELINE:
    command = [sys.executable, '-u', '-m', 'train.dpo.baseline',
               '--source', BASELINE_SOURCE, '--model', BASELINE_MODEL,
               '--quantization', BASELINE_QUANTIZATION, '--compute_dtype', BASELINE_COMPUTE_DTYPE,
               '--batch_size', str(BASELINE_BATCH_SIZE), '--output_dir', str(BASELINE_OUTPUT)]
    if BASELINE_SOURCE == 'local':
        command += ['--local_train_file', str(REPO.parent / 'MedQA-USMLE-4-options/phrases_no_exclude_train.jsonl'),
                    '--local_test_file', str(REPO.parent / 'MedQA-USMLE-4-options/phrases_no_exclude_test.jsonl')]
    run_command(command)
else:
    print('Baseline evaluation disabled; existing results will be displayed if available.')



Generating train split: 100%|██████████| 10178/10178 [00:00<00:00, 61153.88 examples/s]

Generating test split: 100%|██████████| 1273/1273 [00:00<00:00, 130898.48 examples/s]
I0000 00:00:1794234974.067143    4417 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794234974.140315    4417 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1794234976.778920    4417 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set

In [7]:
import pandas as pd
from IPython.display import display
if (BASELINE_OUTPUT / 'summary.json').is_file():
    baseline = json.loads((BASELINE_OUTPUT / 'summary.json').read_text())
    manifest = json.loads((BASELINE_OUTPUT / 'manifest.json').read_text())
    display(pd.DataFrame([{'split': name, 'questions': manifest['counts'][name], **metrics}
                          for name, metrics in baseline.items()]))
    print('Saved baseline configuration:', manifest['model'], manifest['quantization'], manifest['compute_dtype'])
    print('Test scores are descriptive only and must not be used for hyperparameter selection.')
else:
    print('No completed baseline exists. Set RUN_BASELINE=True to evaluate the original model.')


,split,questions,n,accuracy,mean_confidence,ece,brier,nll,hc_wrong_rate,hc_wrong_count,mean_conf_wrong,hc_count,hc_accuracy,mean_letter_mass,min_letter_mass
0,validation,500,500,0.510000,0.953508,0.448445,0.929981,5.053495,0.40400,202,0.945268,431,0.531323,0.999977,0.989966
1,test,1273,1273,0.468185,0.955671,0.487485,1.000214,5.638082,0.43912,559,0.944218,1108,0.495487,0.999986,0.991465


Saved baseline configuration: Qwen/Qwen2.5-3B-Instruct 4bit torch.bfloat16
Test scores are descriptive only and must not be used for hyperparameter selection.


## Run configuration
`PILOT=True` uses 64 pairs, 32 validation questions, and 10 updates in a separate directory. Set it to False for full training. Changing a pilot into a full run requires a new output directory. Use `STOP_AFTER_STEP` instead when pausing a full run while preserving its learning-rate schedule.

Keep the output directory on Google Drive in Colab. Resume requires an existing complete checkpoint and matching arguments.

In [ ]:
SOURCE = 'hf' if IN_COLAB else 'local'
MODEL = 'Qwen/Qwen2.5-3B-Instruct' if SOURCE == 'hf' else str(REPO.parent / 'Qwen2.5-3B-Instruct')
LOCAL_TRAIN_FILE = REPO.parent / 'MedQA-USMLE-4-options/phrases_no_exclude_train.jsonl'
DATA_FILE = REPO / 'outputs/medqa5000_L4_seed42_qpb2/selection_seed42/dpo.jsonl'
PILOT = False
RUN_TRAINING = True
RESUME = False
SEED = 1
QUANTIZATION = '4bit'
EPOCHS = 3
BATCH_SIZE = 4
GRADIENT_ACCUMULATION = 4
MAX_LENGTH = 2048
LEARNING_RATE = 5e-6
BETA = 0.1
LORA_DROPOUT = 0.05
STOP_AFTER_STEP = 0
OUTPUT = REPO / 'outputs/dpo_full_3epochs_seed1_lr5e-6_beta0p1_r16_drop0p05_bs4_ga4'
COMMON = [sys.executable, '-u', '-m', 'train.dpo.train', '--source', SOURCE,
          '--model', MODEL, '--data_file', str(DATA_FILE), '--output_dir', str(OUTPUT),
          '--seed', str(SEED), '--quantization', QUANTIZATION, '--num_train_epochs', str(EPOCHS),
          '--batch_size', str(BATCH_SIZE), '--gradient_accumulation_steps', str(GRADIENT_ACCUMULATION),
          '--max_length', str(MAX_LENGTH), '--learning_rate', str(LEARNING_RATE), '--beta', str(BETA),
          '--lora_dropout', str(LORA_DROPOUT), '--no-baseline_validation']
if SOURCE == 'local':
    COMMON += ['--local_train_file', str(LOCAL_TRAIN_FILE)]
if PILOT:
    COMMON += ['--n_pairs', '64', '--n_val', '32', '--max_steps', '10']
if STOP_AFTER_STEP:
    COMMON += ['--stop_after_step', str(STOP_AFTER_STEP)]
print('Output:', OUTPUT)
print('Training enabled:', RUN_TRAINING)
print('Full run:', not PILOT, 'Epochs:', EPOCHS)

print('Effective batch pairs:', BATCH_SIZE * GRADIENT_ACCUMULATION)


## Data and token-length check
This checks source hashes, question identities, held-out separation, pair labels, and chat-template lengths. It loads the tokenizer, not model weights. Archived raw samples are not needed.

In [ ]:
run_command(COMMON + ['--check_data'])


## Train or resume
Default: NF4 QLoRA, rank 16, alpha 32, active LoRA dropout 0.05, and a fixed base reference. No new responses are sampled. The original-model baseline is handled by the optional stage above and is not repeated here. Adapter validation runs after each epoch. Best-adapter selection uses validation accuracy, then NLL. Test metrics never select a checkpoint. Training checkpoints include optimizer and RNG state; `best_adapter` is for evaluation, not resume.


In [ ]:
if RUN_TRAINING:
    run_command(COMMON + (['--resume'] if RESUME else []))
else:
    print('Training is disabled. Review the configuration and set RUN_TRAINING=True to start.')

## Results
Review training loss separately from medical answer accuracy. ECE, Brier, NLL, and high-confidence errors measure confidence reliability. Timing estimates exclude validation and checkpoint overhead.

In [ ]:
import pandas as pd
from IPython.display import display
summary_file = OUTPUT / 'training_summary.json'
if summary_file.is_file():
    summary = json.loads(summary_file.read_text())
    display(pd.DataFrame([summary['best_validation']['metrics']]))
    print(json.dumps({k: summary[k] for k in ['global_step', 'epoch', 'stopped_early',
          'steady_step_seconds', 'estimated_training_seconds_per_epoch', 'peak_allocated_gib']}, indent=2))
    print('Best adapter:', OUTPUT / 'best_adapter')
else:
    print('No completed training summary exists for this output directory.')
log = OUTPUT / 'training_metrics.jsonl'
if log.is_file():
    history = pd.read_json(log, lines=True)
    display(history.tail(10))
    if 'loss' in history:
        import matplotlib.pyplot as plt
        history.dropna(subset=['loss']).plot(x='step', y='loss', title='DPO training loss', legend=False)
        plt.ylabel('Loss')
        plt.show()

## Final experiments
After validation-based configuration selection, repeat with a second seed and a separate output directory. Evaluate selected adapters on the MedQA test set and transfer datasets using the existing evaluator. Do not use those scores to choose training checkpoints. See `train/dpo/README.txt` for commands and artifact descriptions.

## Post-training evaluation
Training is enabled for the batch-size-4 run in a new output directory. The prior executed notebook is preserved in the sibling backup directory. The one-time baseline remains disabled and its saved results are reused. Open `dpo_evaluate.ipynb` for direct/CoT evaluation and validation-fitted temperature scaling; update its run selection after this experiment completes.